# Experiment 4 — GPT-OSS-20B second-moment (covariance) analysis

This notebook studies **second-moment structure**: whether two concept classes differ in covariance even when their centroids are close. It loads prompted and unprompted GPT-OSS-20B embeddings, reduces them (PCA-50), and compares covariance with the normalised Frobenius norm, eigenvalue spectra, Gaussian KL (mean vs. covariance terms), and permutation tests. Run after Experiments 1 and 2. CPU is sufficient once embeddings exist.

- Outputs have been cleared from the repository version.


In [ ]:
# Repository paths and reproducibility settings
from pathlib import Path
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "llm_data").exists() and (candidate / "llms").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the cloned repository.")

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data" / "llm_data"
RESULTS_ROOT = REPO_ROOT / "results" / "llm"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"LLM data:        {DATA_DIR}")


In [ ]:
# ============================================================
# Second Moment (Covariance) Analysis
# ============================================================
# For classes that are NOT well-separated by Euclidean distance,
# their covariance structure (second moment) may still differ —
# meaning the model encodes something about the distinction,
# just not in the centroid direction.
#
# For each pair we compute and compare:
#
#   FIRST MOMENT (mean-based):
#     1. Centroid Euclidean distance
#     2. Avg pairwise Euclidean inter/intra
#     3. Separability ratio
#
#   SECOND MOMENT (covariance-based):
#     4. Frobenius norm of (Σ_A - Σ_B)      — raw covariance difference
#     5. Eigenvalue spectra comparison        — are classes spread differently?
#     6. Gaussian KL divergence (closed form) — full distributional comparison
#        decomposed into:
#          • Mean term    — contribution from centroid difference
#          • Covariance term — contribution from shape difference
#
# All computed on PCA-50 reduced embeddings (tractable covariance inversion).
# Run on CPU — no GPU needed.
# ============================================================

import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist
from scipy.linalg import logm
import warnings
warnings.filterwarnings("ignore")

# ── Repository result paths ─────────────────────────────────



DIR_UNPR = str(RESULTS_ROOT / "results_pca_gpt_oss_20b_sentences")
DIR_PROM = str(RESULTS_ROOT / "results_pca_gpt_oss_20b_sentences_prompted")
DIR_OUT  = str(RESULTS_ROOT / "second_moment_analysis_gpt_oss_20b")
os.makedirs(DIR_OUT, exist_ok=True)

N_PCA    = 50
ALL_CATS = ["Shakespeare", "Computer Science", "Information Security",
            "Theory of Computation", "Hate Speech", "No-Hate Speech"]
PAIRS    = [
    ("Computer Science",      "Shakespeare"),
    ("Theory of Computation", "Information Security"),
    ("Hate Speech",           "No-Hate Speech"),
]
COLORS = {
    "Shakespeare":           "#E06C75",
    "Computer Science":      "#61AFEF",
    "Information Security":  "#98C379",
    "Theory of Computation": "#E5C07B",
    "Hate Speech":           "#C678DD",
    "No-Hate Speech":        "#56B6C2",
}
SHORT = {
    "Shakespeare": "Shakespeare", "Computer Science": "CS",
    "Information Security": "InfoSec", "Theory of Computation": "ToC",
    "Hate Speech": "Hate", "No-Hate Speech": "No-Hate",
}


# ── Load embeddings ──────────────────────────────────────────
def load_run(folder):
    emb  = np.vstack([np.load(os.path.join(folder, "embeddings.npy")),
                      np.load(os.path.join(folder, "hate_embeddings.npy"))])
    labs = np.concatenate([
        np.load(os.path.join(folder, "labels.npy"),      allow_pickle=True),
        np.load(os.path.join(folder, "hate_labels.npy"), allow_pickle=True),
    ])
    return emb, labs

print("Loading embeddings ...")
emb_u, labs_u = load_run(DIR_UNPR)
emb_p, labs_p = load_run(DIR_PROM)


# ── PCA-50 reduction ─────────────────────────────────────────
def reduce(emb, n=N_PCA):
    n_comp = min(n, emb.shape[0] - 1, emb.shape[1])
    pca    = PCA(n_components=n_comp)
    emb_r  = pca.fit_transform(emb)
    var    = pca.explained_variance_ratio_.sum()
    return emb_r, pca, var

emb_u50, _, var_u = reduce(emb_u)
emb_p50, _, var_p = reduce(emb_p)
print(f"  Unprompted PCA-{N_PCA} variance explained: {var_u:.1%}")
print(f"  Prompted   PCA-{N_PCA} variance explained: {var_p:.1%}")


# ── Gaussian KL divergence (closed form) ─────────────────────
# KL(N(μ1,Σ1) ‖ N(μ2,Σ2)) = 0.5 * [tr(Σ2⁻¹Σ1) + (μ2-μ1)ᵀΣ2⁻¹(μ2-μ1) - k + ln(det(Σ2)/det(Σ1))]
# Decomposed into:
#   mean_term = 0.5 * (μ2-μ1)ᵀ Σ2⁻¹ (μ2-μ1)
#   cov_term  = 0.5 * [tr(Σ2⁻¹Σ1) - k + ln(det(Σ2)/det(Σ1))]
# We return symmetric version: 0.5*(KL(A‖B) + KL(B‖A))

def gaussian_kl(mu1, S1, mu2, S2):
    k    = len(mu1)
    reg  = np.eye(k) * 1e-6
    S1r, S2r = S1 + reg, S2 + reg
    try:
        S2_inv = np.linalg.inv(S2r)
    except np.linalg.LinAlgError:
        S2_inv = np.linalg.pinv(S2r)

    diff      = mu2 - mu1
    mean_term = 0.5 * float(diff @ S2_inv @ diff)
    sign1, ld1 = np.linalg.slogdet(S1r)
    sign2, ld2 = np.linalg.slogdet(S2r)
    cov_term  = 0.5 * (np.trace(S2_inv @ S1r) - k + (ld2 - ld1))
    kl_12     = mean_term + cov_term

    # reverse direction
    try:
        S1_inv = np.linalg.inv(S1r)
    except np.linalg.LinAlgError:
        S1_inv = np.linalg.pinv(S1r)
    diff2      = mu1 - mu2
    mean_term2 = 0.5 * float(diff2 @ S1_inv @ diff2)
    cov_term2  = 0.5 * (np.trace(S1_inv @ S2r) - k + (ld1 - ld2))
    kl_21      = mean_term2 + cov_term2

    sym_kl       = 0.5 * (kl_12 + kl_21)
    sym_mean     = 0.5 * (mean_term + mean_term2)
    sym_cov      = 0.5 * (cov_term + cov_term2)
    return sym_kl, sym_mean, sym_cov


# ── Permutation test for Frobenius norm ──────────────────────
# H0: the two classes have the same covariance (any observed difference
#     is due to sampling noise from a shared population).
# Procedure: pool both classes, randomly split into same-sized groups
# n_perm times, compute normalised Frobenius each time → null distribution.
# p-value = fraction of null samples >= observed value.

def permutation_frobenius_test(d1, d2, n_perm=1000, seed=42):
    rng      = np.random.default_rng(seed)
    n1       = len(d1)
    combined = np.vstack([d1, d2])
    n_total  = len(combined)

    def norm_frob(a, b):
        Sa = np.cov(a, rowvar=False)
        Sb = np.cov(b, rowvar=False)
        diff    = np.linalg.norm(Sa - Sb, ord="fro")
        avg_n   = 0.5 * (np.linalg.norm(Sa, ord="fro") + np.linalg.norm(Sb, ord="fro"))
        return diff / avg_n if avg_n > 0 else 0.0

    observed  = norm_frob(d1, d2)
    null_dist = np.array([
        norm_frob(combined[perm := rng.permutation(n_total)][:n1],
                  combined[perm][n1:])
        for _ in range(n_perm)
    ])
    p_value   = float(np.mean(null_dist >= observed))
    return observed, null_dist, p_value


def plot_null_distribution(observed, null_dist, p_value, c1, c2, tag, out_dir):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(null_dist, bins=40, color="#61AFEF", alpha=0.75, edgecolor="white",
            label=f"Null distribution (n={len(null_dist)} permutations)")
    ax.axvline(observed, color="#E06C75", linewidth=2,
               label=f"Observed = {observed:.4f}  (p = {p_value:.3f})")
    ax.set_xlabel("Normalised Frobenius norm", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.set_title(f"Permutation Test: Covariance Difference\n"
                 f"{SHORT[c1]} vs {SHORT[c2]}  [{tag}]", fontsize=12, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    safe  = f"{c1}_{c2}_{tag}".lower().replace(" ", "_")
    fpath = os.path.join(out_dir, f"permtest_{safe}.png")
    plt.savefig(fpath, dpi=150, bbox_inches="tight")
    print(f"  Saved {fpath}")
    plt.show(); plt.close()


# ── Per-pair analysis ────────────────────────────────────────
def analyse_pair(emb_full, emb50, labs, c1, c2, tag, report):
    d1_full = emb_full[labs == c1]
    d2_full = emb_full[labs == c2]
    d1      = emb50[labs == c1]
    d2      = emb50[labs == c2]
    mu1, mu2 = d1.mean(axis=0), d2.mean(axis=0)
    S1       = np.cov(d1, rowvar=False)
    S2       = np.cov(d2, rowvar=False)

    # ── First moment metrics ─────────────────────────────────
    centroid_dist = np.linalg.norm(mu1 - mu2)

    def avg_intra(d):
        n = d.shape[0]; dmat = cdist(d, d, "euclidean")
        triu = np.triu_indices(n, k=1)
        return dmat[triu].mean() if len(triu[0]) > 0 else 1.0

    inter_euc = cdist(d1_full, d2_full, "euclidean").mean()
    intra_avg = np.mean([avg_intra(d1_full), avg_intra(d2_full)])
    sep_ratio = inter_euc / intra_avg if intra_avg > 0 else float("inf")

    # ── Second moment metrics ────────────────────────────────
    frob_diff = np.linalg.norm(S1 - S2, ord="fro")

    # Normalised Frobenius: divide by average of the two norms
    avg_norm  = 0.5 * (np.linalg.norm(S1, ord="fro") + np.linalg.norm(S2, ord="fro"))
    frob_norm = frob_diff / avg_norm if avg_norm > 0 else 0.0

    # Gaussian KL decomposition
    kl_sym, kl_mean, kl_cov = gaussian_kl(mu1, S1, mu2, S2)
    cov_fraction = kl_cov / kl_sym if kl_sym > 0 else 0.0

    # Permutation test on normalised Frobenius
    print(f"  Running permutation test: {c1} vs {c2} [{tag}] ...")
    _, null_dist, p_frob = permutation_frobenius_test(d1, d2)
    plot_null_distribution(frob_norm, null_dist, p_frob, c1, c2, tag, DIR_OUT)

    report.append(f"\n  {'─'*60}")
    report.append(f"  PAIR: {c1}  vs  {c2}  [{tag}]")
    report.append(f"  {'─'*60}")
    report.append(f"\n  FIRST MOMENT (mean-based, full {emb_full.shape[1]}-d):")
    report.append(f"    Centroid Euclidean distance : {centroid_dist:.4f}  (PCA-{N_PCA} space)")
    report.append(f"    Avg pairwise inter-class    : {inter_euc:.4f}")
    report.append(f"    Avg pairwise intra-class    : {intra_avg:.4f}")
    report.append(f"    Separability ratio          : {sep_ratio:.4f}")

    report.append(f"\n  SECOND MOMENT (covariance-based, PCA-{N_PCA} space):")
    report.append(f"    Frobenius ‖Σ₁ - Σ₂‖_F      : {frob_diff:.4f}")
    report.append(f"    Normalised Frobenius         : {frob_norm:.4f}  (÷ avg ‖Σ‖_F)")
    report.append(f"    Permutation test p-value     : {p_frob:.3f}  "
                  f"({'significant at p<0.05' if p_frob < 0.05 else 'not significant at p<0.05'})")
    report.append(f"    Symmetric Gaussian KL        : {kl_sym:.4f}")
    report.append(f"      ↳ Mean term  contribution : {kl_mean:.4f}  ({kl_mean/kl_sym*100:.1f}%)")
    report.append(f"      ↳ Covariance contribution : {kl_cov:.4f}  ({cov_fraction*100:.1f}%)")

    # Observation flags — describe what is found, not what it means
    if sep_ratio < 1.1 and frob_norm > 0.3:
        report.append(f"\n  [OBS] Low Euclidean separability (ratio={sep_ratio:.3f}) with "
                       f"high covariance difference (norm={frob_norm:.3f})")
    elif sep_ratio >= 1.1 and frob_norm > 0.3:
        report.append(f"\n  [OBS] Both Euclidean and covariance differences are present "
                       f"(ratio={sep_ratio:.3f}, frob={frob_norm:.3f})")
    elif sep_ratio < 1.1 and frob_norm <= 0.3:
        report.append(f"\n  [OBS] Both Euclidean and covariance differences are small "
                       f"(ratio={sep_ratio:.3f}, frob={frob_norm:.3f})")
    else:
        report.append(f"\n  [OBS] Euclidean separability present with modest covariance difference "
                       f"(ratio={sep_ratio:.3f}, frob={frob_norm:.3f})")

    if kl_cov > kl_mean:
        report.append(f"  [OBS] Covariance term dominates Gaussian KL "
                       f"({cov_fraction:.0%} of total KL={kl_sym:.4f})")
    else:
        report.append(f"  [OBS] Mean term dominates Gaussian KL "
                       f"({kl_mean/kl_sym*100:.0f}% of total KL={kl_sym:.4f})")

    return {
        "centroid_dist": centroid_dist, "sep_ratio": sep_ratio,
        "frob_norm": frob_norm, "p_frob": p_frob, "kl_sym": kl_sym,
        "kl_mean": kl_mean, "kl_cov": kl_cov,
        "S1": S1, "S2": S2, "d1": d1, "d2": d2,
        "mu1": mu1, "mu2": mu2,
    }


# ── Eigenvalue spectra plot ──────────────────────────────────
def plot_eigenspectra(res, c1, c2, tag, out_dir, top_k=20):
    evals1 = np.sort(np.linalg.eigvalsh(res["S1"]))[::-1][:top_k]
    evals2 = np.sort(np.linalg.eigvalsh(res["S2"]))[::-1][:top_k]

    x = np.arange(top_k)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: eigenvalue bar chart
    ax = axes[0]
    w  = 0.35
    ax.bar(x - w/2, evals1, w, label=SHORT[c1], color=COLORS[c1], alpha=0.85)
    ax.bar(x + w/2, evals2, w, label=SHORT[c2], color=COLORS[c2], alpha=0.85)
    ax.set_xlabel("Eigenvalue rank", fontsize=11)
    ax.set_ylabel("Eigenvalue magnitude", fontsize=11)
    ax.set_title(f"Covariance Eigenvalue Spectra\n{SHORT[c1]} vs {SHORT[c2]}  [{tag}]",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.4)

    # Right: cumulative variance explained within each class
    ax2 = axes[1]
    cum1 = np.cumsum(evals1) / np.sum(np.linalg.eigvalsh(res["S1"]))
    cum2 = np.cumsum(evals2) / np.sum(np.linalg.eigvalsh(res["S2"]))
    ax2.plot(x + 1, cum1, "o-", color=COLORS[c1], label=SHORT[c1], linewidth=2)
    ax2.plot(x + 1, cum2, "s-", color=COLORS[c2], label=SHORT[c2], linewidth=2)
    ax2.set_xlabel("Number of eigenvectors", fontsize=11)
    ax2.set_ylabel("Cumulative variance explained", fontsize=11)
    ax2.set_title(f"Cumulative Within-Class Variance\n{SHORT[c1]} vs {SHORT[c2]}  [{tag}]",
                  fontsize=12, fontweight="bold")
    ax2.legend(fontsize=10)
    ax2.grid(True, linestyle="--", alpha=0.4)

    plt.tight_layout()
    safe  = f"{c1}_{c2}_{tag}".lower().replace(" ", "_")
    fpath = os.path.join(out_dir, f"eigenspectra_{safe}.png")
    plt.savefig(fpath, dpi=150, bbox_inches="tight")
    print(f"  Saved {fpath}")
    plt.show(); plt.close()


# ── Summary comparison bar chart ─────────────────────────────
def plot_summary(all_results, out_dir):
    metrics  = ["sep_ratio", "frob_norm", "kl_mean %", "kl_cov %"]
    n_pairs  = len(PAIRS)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes     = axes.flatten()

    for ax_idx, metric in enumerate(metrics):
        ax     = axes[ax_idx]
        labels = [f"{SHORT[c1]} vs\n{SHORT[c2]}" for c1, c2 in PAIRS]
        x      = np.arange(n_pairs)
        w      = 0.35

        for run_idx, (tag, color) in enumerate([("Unprompted", "#61AFEF"), ("Prompted", "#E06C75")]):
            vals = []
            for c1, c2 in PAIRS:
                r = all_results[(c1, c2, tag)]
                if metric == "sep_ratio":
                    vals.append(r["sep_ratio"])
                elif metric == "frob_norm":
                    vals.append(r["frob_norm"])
                elif metric == "kl_mean %":
                    total = r["kl_sym"]
                    vals.append(r["kl_mean"] / total * 100 if total > 0 else 0)
                elif metric == "kl_cov %":
                    total = r["kl_sym"]
                    vals.append(r["kl_cov"] / total * 100 if total > 0 else 0)
            ax.bar(x + run_idx * w - w/2, vals, w, label=tag, color=color, alpha=0.85, edgecolor="white")

        titles = {
            "sep_ratio":  "Separability Ratio\n(inter / avg intra Euclidean)  — higher = more separated",
            "frob_norm":  "Normalised Frobenius ‖Σ₁-Σ₂‖_F\n(covariance difference)  — higher = more different shape",
            "kl_mean %":  "Gaussian KL: Mean Term %\n(how much centroid difference drives KL)",
            "kl_cov %":   "Gaussian KL: Covariance Term %\n(how much shape difference drives KL)",
        }
        ax.set_title(titles[metric], fontsize=11, fontweight="bold")
        ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=10)
        ax.legend(fontsize=10)
        ax.grid(True, linestyle="--", alpha=0.4)
        if metric == "sep_ratio":
            ax.axhline(1.0, color="red", linestyle="--", linewidth=1, label="ratio=1 (no separation)")

    plt.suptitle("First vs Second Moment Separation — GPT-OSS-20B\n"
                 "Unprompted vs Prompted", fontsize=13, fontweight="bold")
    plt.tight_layout()
    fpath = os.path.join(out_dir, "summary_first_vs_second_moment.png")
    plt.savefig(fpath, dpi=150, bbox_inches="tight")
    print(f"  Saved {fpath}")
    plt.show(); plt.close()


# ── Run all analyses ─────────────────────────────────────────
all_results = {}
report      = ["=" * 70,
               "FIRST vs SECOND MOMENT ANALYSIS — GPT-OSS-20B",
               f"PCA-{N_PCA} space for covariance metrics",
               "=" * 70]

for tag, emb_full, emb50, labs in [
    ("Unprompted", emb_u, emb_u50, labs_u),
    ("Prompted",   emb_p, emb_p50, labs_p),
]:
    report.append(f"\n{'═'*70}")
    report.append(f"  {tag.upper()}")
    report.append(f"{'═'*70}")

    for c1, c2 in PAIRS:
        res = analyse_pair(emb_full, emb50, labs, c1, c2, tag, report)
        all_results[(c1, c2, tag)] = res
        plot_eigenspectra(res, c1, c2, tag, DIR_OUT)

report.append(f"\n{'='*70}")
report.append("  DATA-DRIVEN SUMMARY")
report.append(f"{'='*70}\n")

for c1, c2 in PAIRS:
    pair_label = f"{SHORT[c1]} vs {SHORT[c2]}"
    r_u = all_results[(c1, c2, "Unprompted")]
    r_p = all_results[(c1, c2, "Prompted")]

    report.append(f"  {pair_label}:")

    for tag, r in [("Unprompted", r_u), ("Prompted", r_p)]:
        euc_level = ("strong" if r["sep_ratio"] > 1.5
                     else "moderate" if r["sep_ratio"] > 1.1
                     else "weak")
        cov_level = ("large" if r["frob_norm"] > 0.5
                     else "moderate" if r["frob_norm"] > 0.2
                     else "small")
        dominant  = ("covariance" if r["kl_cov"] > r["kl_mean"] else "mean")
        sig_str   = f"p={r['p_frob']:.3f} {'*' if r['p_frob'] < 0.05 else '(ns)'}"
        report.append(
            f"    [{tag}] Euclidean sep={euc_level} (ratio={r['sep_ratio']:.3f}), "
            f"covariance diff={cov_level} (frob={r['frob_norm']:.3f}, {sig_str}), "
            f"KL dominated by {dominant} term "
            f"({r['kl_mean']/r['kl_sym']*100:.0f}% mean / {r['kl_cov']/r['kl_sym']*100:.0f}% cov)"
        )

    # Effect of prompting on each metric
    d_sep  = r_p["sep_ratio"] - r_u["sep_ratio"]
    d_frob = r_p["frob_norm"] - r_u["frob_norm"]
    d_kl   = r_p["kl_sym"]   - r_u["kl_sym"]
    report.append(
        f"    [Δ Prompted vs Unprompted] "
        f"sep_ratio {d_sep:+.3f},  frob_norm {d_frob:+.3f},  kl_sym {d_kl:+.4f}"
    )
    report.append("")

txt = "\n".join(report)
print(txt)
with open(os.path.join(DIR_OUT, "second_moment_report.txt"), "w", encoding="utf-8") as f:
    f.write(txt)
print(f"Saved second_moment_report.txt")

plot_summary(all_results, DIR_OUT)

print(f"\n{'='*60}")
print(f"All results saved to: {DIR_OUT}")
print(f"{'='*60}")
